# Configuración de ElectroCasa

Crear los grupos de Ingeniería, Analistas y Auditoría desde **Settings → Identity and access → Groups** y agregar como miembro de Ingeniería al dueño del pipeline. Ejecutar este notebook una vez en cada catálogo que vaya a desplegarse. Usar el modo de usuario únicamente para la alternativa de Free Edition si Unity Catalog rechaza los grupos, y guardar ese error como evidencia.

In [ ]:
# Selecciona el target y el modo de permisos antes de crear objetos.
dbutils.widgets.dropdown("catalog", "electrocasa_dev", ["electrocasa_dev", "electrocasa_prod"])
dbutils.widgets.dropdown("modo_usuario", "false", ["false", "true"])
dbutils.widgets.dropdown("solo_objetos", "false", ["false", "true"])
solo_objetos = dbutils.widgets.get("solo_objetos") == "true"
catalog = dbutils.widgets.get("catalog")
if catalog not in {"electrocasa_dev", "electrocasa_prod"}:
    raise ValueError("Seleccione un catálogo de ElectroCasa.")
fallback = dbutils.widgets.get("modo_usuario") == "true"
account = spark.sql("SELECT current_user() AS email").first().email
if fallback:
    engineer = "`" + account.replace("`", "``") + "`"
    analyst = auditor = "`account users`"
    identity = "current_user() = '" + account.replace("'", "''") + "'"
else:
    engineer = "`electrocasa_ingenieria`"
    analyst = "`electrocasa_analistas`"
    auditor = "`electrocasa_auditoria`"
    identity = "is_account_group_member('electrocasa_ingenieria')" 

In [ ]:
# Crea los objetos administrados y las carpetas de landing.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
for schema in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.default.vol_landing")
for source in ("ventas", "empleados", "devoluciones", "resenas", "catalogo", "sucursales"):
    dbutils.fs.mkdirs(f"/Volumes/{catalog}/default/vol_landing/{source}")

# Define el esquema de las dos tablas Delta de COPY INTO.
reference = {
    "catalogo": "producto_id STRING, nombre_producto STRING, categoria STRING, marca STRING, "
                "precio_lista STRING, activo STRING",
    "sucursales": "sucursal_id STRING, nombre_sucursal STRING, distrito STRING, ciudad STRING, "
                  "region STRING, fecha_apertura STRING",
}
for name, fields in reference.items():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.bronze.{name} (
            {fields}, _archivo STRING, _lote STRING, _ingestado_en TIMESTAMP
        ) USING DELTA
    """)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.correcciones (
        fuente STRING NOT NULL, registro_id STRING NOT NULL, revision BIGINT NOT NULL,
        payload_json STRING NOT NULL, motivo STRING, registrado_en TIMESTAMP
    ) USING DELTA
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog}.silver.cuarentena_historial (
        run_id BIGINT, registro_id STRING, fuente STRING, lote STRING, archivo STRING,
        regla STRING, procesado_en TIMESTAMP
    ) USING DELTA
""")

# Instala las máscaras que el pipeline aplica en Silver y Gold.
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog}.silver.mask_dni(valor STRING)
    RETURNS STRING RETURN CASE WHEN {identity} THEN valor WHEN valor IS NOT NULL THEN '***' END
""")
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog}.silver.mask_salario(valor DECIMAL(12,2))
    RETURNS DECIMAL(12,2) RETURN CASE WHEN {identity} THEN valor END
""")

In [ ]:
# Aplica los permisos cuando ya existen los grupos del examen.
if not solo_objetos:
    # Aplica permisos heredables del catálogo, esquemas y volumen.
    for principal in {engineer, analyst, auditor}:
        spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO {principal}")
    for schema in ("bronze", "silver", "gold"):
        spark.sql(f"GRANT USE SCHEMA, SELECT, MODIFY ON SCHEMA {catalog}.{schema} TO {engineer}")
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.default TO {engineer}")
    spark.sql(f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME {catalog}.default.vol_landing TO {engineer}")
    for principal in {analyst, auditor}:
        spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {catalog}.gold TO {principal}")

    # Comprueba la membresía con la identidad que utilizará el despliegue.
    if not fallback and not spark.sql(f"SELECT {identity} AS es_ingenieria").first().es_ingenieria:
        raise RuntimeError("La cuenta ejecutora debe pertenecer al grupo de Ingeniería.")

## Evidencia posterior al despliegue

Ejecutar las consultas de gobierno del README después de cargar los datos. Para conceder acceso a auditoría sobre las tablas de sistema se requieren privilegios de cuenta. Si Free Edition rechaza estos grants, conservar el error y usar la pestaña **Lineage** de Catalog Explorer.